# 03: Arc-DSL Task Analysis Debug Notebook

This notebook analyzes all 400 neurogolf tasks using the arc-dsl solver library.
- Maps each task to a pre-written DSL solver
- Extracts DSL primitives used per task
- Verifies solutions on training examples
- Outputs debug info and method summaries

In [ ]:
import json
import sys
import os
import re
from collections import Counter, defaultdict
import inspect

sys.path.insert(0, os.path.join(os.getcwd(), '..', 'arc-dsl'))

import solvers as S
from dsl import *

# Load analysis results
with open('task_analysis.json') as f:
    data = json.load(f)

print(f"Total tasks: {data['summary']['total_tasks']}")
print(f"Matched: {data['summary']['matched_tasks']}")
print(f"Verification passed: {data['summary']['verification_passed']}")

## Section 1: Primitive Frequency Analysis

In [ ]:
# Top primitives
print("=" * 60)
print("TOP 30 DSL PRIMITIVES BY USAGE")
print("=" * 60)
for func, count in list(data['primitive_frequency'].items())[:30]:
    bar = "#" * (count // 5)
    print(f"{func:20s} {count:4d} {bar}")

## Section 2: Task Categories by Grid Size

In [ ]:
# Categorize tasks by input/output size relationship
same_size = []
larger_output = []
smaller_output = []

for k, v in data['tasks'].items():
    if v['grid_info']['train_examples']:
        ex = v['grid_info']['train_examples'][0]
        inp_parts = list(map(int, ex['input_shape'].split('x')))
        out_parts = list(map(int, ex['output_shape'].split('x')))
        inp_area = inp_parts[0] * inp_parts[1]
        out_area = out_parts[0] * out_parts[1]
        
        if inp_area == out_area:
            same_size.append(k)
        elif out_area > inp_area:
            larger_output.append(k)
        else:
            smaller_output.append(k)

print(f"Same size: {len(same_size)} tasks")
print(f"Larger output: {len(larger_output)} tasks")
print(f"Smaller output: {len(smaller_output)} tasks")

## Section 3: Specific Task Inspector

In [ ]:
# Inspect a specific task
TASK_NUM = 1
task_key = f"task{TASK_NUM:03d}"
task_info = data['tasks'][task_key]

print(f"Task: {task_key}")
print(f"Solver: {task_info['solver']}")
if task_info['primitives']:
    print(f"Functions: {task_info['primitives']['functions']}")
    print(f"Constants: {task_info['primitives']['constants']}")
    print(f"\nSource preview:")
    print(task_info['primitives']['source_preview'])

print(f"\nVerification:")
for r in (task_info['verification'] or []):
    print(f"  Example {r.get('example', '?')}: match={r.get('match', 'N/A')}")

## Section 4: Primitives by Task Category

In [ ]:
# Analyze which primitives are used for same-size vs different-size tasks
same_size_primitives = Counter()
diff_size_primitives = Counter()

for k, v in data['tasks'].items():
    if not v['primitives']:
        continue
    
    ex = v['grid_info']['train_examples'][0]
    same = ex['same_size']
    
    for func in v['primitives']['functions']:
        if same:
            same_size_primitives[func] += 1
        else:
            diff_size_primitives[func] += 1

print("=" * 60)
print("PRIMITIVES USED FOR SAME-SIZE TASKS")
print("=" * 60)
for func, count in same_size_primitives.most_common(15):
    print(f"  {func:20s} {count:4d}")

print("\n" + "=" * 60)
print("PRIMITIVES USED FOR DIFFERENT-SIZE TASKS")
print("=" * 60)
for func, count in diff_size_primitives.most_common(15):
    print(f"  {func:20s} {count:4d}")

## Section 5: Verification Failures

In [ ]:
# Show tasks with verification failures
print("Tasks with verification failures:")
for k, v in data['tasks'].items():
    if v['verification']:
        for r in v['verification']:
            if not r.get('match', True):
                print(f"  {k}: solver={v['solver']}, example={r.get('example')}")

print("\nUnmatched tasks:")
for k, v in data['tasks'].items():
    if v['solver'] == 'NO_MATCH':
        print(f"  {k}")

## Section 6: DSL Primitive Categories

In [ ]:
# Categorize primitives by type
categories = {
    'Transform (rotate/mirror)': ['rot90', 'rot180', 'rot270', 'hmirror', 'vmirror', 'dmirror', 'cmirror'],
    'Grid operations': ['crop', 'hconcat', 'vconcat', 'hsplit', 'vsplit', 'tophalf', 'bottomhalf', 'lefthalf', 'righthalf'],
    'Color operations': ['replace', 'switch', 'fill', 'underfill', 'recolor', 'paint', 'cover'],
    'Object detection': ['objects', 'colorfilter', 'sizefilter', 'mfilter', 'sfilter', 'partition', 'fgpartition'],
    'Spatial queries': ['ofcolor', 'asindices', 'asobject', 'frontiers', 'delta', 'box', 'border'],
    'Higher-order': ['apply', 'mapply', 'compose', 'fork', 'chain', 'rbind', 'lbind', 'sfilter'],
    'Geometry': ['ulcorner', 'urcorner', 'llcorner', 'lrcorner', 'center', 'height', 'width', 'shape', 'size'],
    'Movement': ['move', 'shift', 'gravitate', 'shoot'],
    'Color queries': ['mostcolor', 'leastcolor', 'palette', 'color', 'colorfilter']
}

print("=" * 60)
print("PRIMITIVE USAGE BY CATEGORY")
print("=" * 60)
for cat_name, funcs in categories.items():
    total = sum(data['primitive_frequency'].get(f, 0) for f in funcs)
    print(f"\n{cat_name} ({total} total uses):")
    for f in funcs:
        count = data['primitive_frequency'].get(f, 0)
        if count > 0:
            print(f"  {f:20s} {count:4d}")

## Section 7: Export Summary

In [ ]:
# Create a compact summary for strategy development
summary = {
    'total_tasks': 400,
    'matched_with_solver': data['summary']['matched_tasks'],
    'unmatched': data['summary']['unmatched_tasks'],
    'top_primitives': dict(list(data['primitive_frequency'].items())[:20]),
    'unique_solvers_used': len(set(v['solver'] for v in data['tasks'].values() if v['solver'] != 'NO_MATCH')),
}

with open('summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print("Summary exported to summary.json")
print(json.dumps(summary, indent=2))